# Apriori Refined Final (Legacy)

**Source script:** `apriori_refined_final.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import re
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from run_eda_v3 import (
    SEED,
    add_moon_cyclic_features,
    build_feature_blocks,
    classify_columns,
    derive_temporal_columns,
    infer_date_column,
    infer_outcome_column,
    load_dataset,
)

ROOT = Path(__file__).resolve().parent
DEFAULT_INPUT = ROOT / "outputs" / "imputed_dataset_enriched.csv"
DEFAULT_OUTDIR = ROOT / "outputs" / "eda_v3" / "apriori_refined_final"

SYNODIC_MONTH_DAYS = 29.530588


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Controlled environmental Apriori pass (final).")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT, help="Input enriched dataset")
    parser.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR, help="Output directory")
    parser.add_argument("--seed", type=int, default=SEED, help="Random seed")
    parser.add_argument("--corr-threshold", type=float, default=0.85, help="Correlation threshold (same logic as modeling)")
    parser.add_argument("--window-days", type=float, default=2.0, help="Lunar near-window days")
    parser.add_argument(
        "--interpretation-workbook",
        type=Path,
        default=Path("/Users/omaralneser/Desktop/3YP/Data_set_cats for analysis.xlsx"),
        help="Workbook containing hematology interpretation thresholds.",
    )
    parser.add_argument(
        "--interpretation-sheet",
        type=str,
        default="Heamtology parameter interpreta",
        help="Sheet name with hematology interpretation thresholds.",
    )
    return parser.parse_args()


def normalize_name(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def sanitize_item_name(text: str) -> str:
    out = re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")
    return out or "unknown"


def load_hematology_interpretation(workbook: Path, sheet_name: str) -> pd.DataFrame:
    ref = pd.read_excel(workbook, sheet_name=sheet_name)
    cols = {c.lower().strip(): c for c in ref.columns}
    param_col = cols.get("haematology parameter")
    interp_col = cols.get("interpretation")
    min_col = cols.get("min")
    max_col = cols.get("max")
    if min_col is None:
        min_col = cols.get("min ")
    if max_col is None:
        max_col = cols.get("max ")
    if any(v is None for v in [param_col, interp_col, min_col, max_col]):
        raise ValueError("Interpretation sheet missing required columns: Haematology parameter, Interpretation, Min, Max")

    out = ref[[param_col, interp_col, min_col, max_col]].copy()
    out.columns = ["parameter", "interpretation", "min_val", "max_val"]
    out["parameter"] = out["parameter"].astype(str).str.strip()
    out["interpretation"] = out["interpretation"].astype(str).str.strip()
    out["min_val"] = pd.to_numeric(out["min_val"], errors="coerce")
    out["max_val"] = pd.to_numeric(out["max_val"], errors="coerce")
    return out


def find_ref_parameter(ref_df: pd.DataFrame, patterns: Sequence[str]) -> str | None:
    params = sorted(ref_df["parameter"].dropna().astype(str).unique().tolist())
    for p in params:
        n = normalize_name(p)
        if any(pt in n for pt in patterns):
            return p
    return None


def extract_threshold_from_ref(ref_df: pd.DataFrame, parameter: str, direction: str) -> float | None:
    sub = ref_df[ref_df["parameter"] == parameter].copy()
    if sub.empty:
        return None
    sub["interp_norm"] = sub["interpretation"].map(normalize_name)

    if direction == "high":
        preferred = [
            "increased",
            "pancreatitislikely",
            "possiblehyperthyreose",
            "pancreatitispossible",
            "upperlimit",
        ]
        for key in preferred:
            m = sub[sub["interp_norm"] == key]
            if not m.empty:
                v = m["min_val"].dropna()
                if not v.empty:
                    return float(v.min())
        v = sub["min_val"].dropna()
        return float(v.max()) if not v.empty else None

    preferred_low = ["lowerlimit", "subnormal"]
    for key in preferred_low:
        m = sub[sub["interp_norm"] == key]
        if not m.empty:
            v = m["max_val"].dropna()
            if not v.empty:
                return float(v.max())
            v2 = m["min_val"].dropna()
            if not v2.empty:
                return float(v2.max())
    v = sub["max_val"].dropna()
    return float(v.min()) if not v.empty else None


def correlation_filter_like_modeling(df: pd.DataFrame, weather_numeric_cols: Sequence[str], threshold: float) -> List[str]:
    if not weather_numeric_cols:
        return []
    X = df[list(weather_numeric_cols)].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    corr = X.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]
    return [c for c in weather_numeric_cols if c not in to_drop]


def build_model_aligned_blocks(df: pd.DataFrame, corr_threshold: float) -> Tuple[pd.DataFrame, str, Dict[str, Dict[str, List[str]]], List[str]]:
    outcome_col = infer_outcome_column(df)
    date_col = infer_date_column(df)
    df2, temporal_cols = derive_temporal_columns(df, date_col)
    df2, _ = add_moon_cyclic_features(df2)

    col_groups = classify_columns(df2, outcome_col, temporal_cols)
    kept_weather_numeric = correlation_filter_like_modeling(
        df=df2,
        weather_numeric_cols=col_groups["weather_model_numeric"],
        threshold=corr_threshold,
    )

    feature_blocks = build_feature_blocks(col_groups)
    env_numeric_original = set(col_groups["weather_model_numeric"])
    feature_blocks["environmental_only"]["numeric"] = sorted(kept_weather_numeric)
    feature_blocks["combined"]["numeric"] = sorted(
        [c for c in feature_blocks["combined"]["numeric"] if c not in env_numeric_original]
        + list(kept_weather_numeric)
    )

    return df2, outcome_col, feature_blocks, kept_weather_numeric


def circular_distance_deg(angle_deg: np.ndarray, anchor_deg: float) -> np.ndarray:
    d = np.abs(angle_deg - anchor_deg)
    return np.minimum(d, 360.0 - d)


def add_lunar_window_flags(df: pd.DataFrame, window_days: float) -> pd.DataFrame:
    out = df.copy()
    window_deg = float(window_days) * (360.0 / SYNODIC_MONTH_DAYS)

    angle_col = None
    for c in out.columns:
        if normalize_name(c) == "moonphaseangledeg":
            angle_col = c
            break

    if angle_col is not None:
        angle = pd.to_numeric(out[angle_col], errors="coerce").to_numpy(dtype=float)
    elif {"moon_phase_sin", "moon_phase_cos"}.issubset(out.columns):
        radians = np.arctan2(
            pd.to_numeric(out["moon_phase_sin"], errors="coerce").to_numpy(dtype=float),
            pd.to_numeric(out["moon_phase_cos"], errors="coerce").to_numpy(dtype=float),
        )
        angle = np.degrees(radians)
    else:
        angle = np.full(len(out), np.nan, dtype=float)

    angle = np.mod(angle, 360.0)
    near_new = circular_distance_deg(angle, 0.0) <= window_deg
    near_full = circular_distance_deg(angle, 180.0) <= window_deg

    out["moon_near_new"] = pd.Series(near_new, index=out.index).fillna(False).astype(bool)
    out["moon_near_full"] = pd.Series(near_full, index=out.index).fillna(False).astype(bool)
    return out


def build_environmental_items(
    df: pd.DataFrame,
    env_numeric_cols: Sequence[str],
    env_categorical_cols: Sequence[str],
) -> pd.DataFrame:
    items: Dict[str, pd.Series] = {}

    for c in env_numeric_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        valid = s.dropna()
        if valid.empty:
            continue
        q25 = valid.quantile(0.25)
        q75 = valid.quantile(0.75)
        if pd.isna(q25) or pd.isna(q75) or float(q25) >= float(q75):
            continue
        low_name = f"{sanitize_item_name(c)}_low"
        high_name = f"{sanitize_item_name(c)}_high"
        low_flag = (s <= q25).fillna(False)
        high_flag = (s >= q75).fillna(False)
        if bool(low_flag.any()):
            items[low_name] = low_flag.astype(bool)
        if bool(high_flag.any()):
            items[high_name] = high_flag.astype(bool)

    for c in env_categorical_cols:
        s = df[c].astype(str).str.strip()
        s = s.replace({"": "missing", "nan": "missing", "None": "missing", "NaN": "missing"}).fillna("missing")
        vc = s.value_counts(dropna=False)
        for val in vc.index.tolist():
            item_name = f"{sanitize_item_name(c)}__{sanitize_item_name(val)}"
            items[item_name] = (s == val).astype(bool)

    for moon_flag in ["moon_near_full", "moon_near_new"]:
        if moon_flag in df.columns:
            items[moon_flag] = df[moon_flag].fillna(False).astype(bool)

    tx = pd.DataFrame(items, index=df.index).fillna(False).astype(bool)
    return tx


def pick_outcome_label(labels: Sequence[str], patterns: Sequence[str]) -> str | None:
    for lab in labels:
        n = normalize_name(lab)
        if any(p in n for p in patterns):
            return lab
    return None


def build_disease_consequents(df: pd.DataFrame, outcome_col: str) -> Tuple[pd.DataFrame, Dict[str, float]]:
    y = df[outcome_col].astype(str)
    labels = sorted(y.unique().tolist())
    mapping = {
        "disease_ap": ["pancreat"],
        "disease_tri": ["triad", "chronicenteropath"],
        "disease_ge": ["gastroenteritis"],
        "disease_parvo": ["parvo"],
    }

    data: Dict[str, pd.Series] = {}
    baseline: Dict[str, float] = {}
    for key, pats in mapping.items():
        lab = pick_outcome_label(labels, pats)
        if lab is None:
            continue
        flag = (y == lab).astype(bool)
        data[key] = flag
        baseline[key] = float(flag.mean())

    return pd.DataFrame(data, index=df.index), baseline


def compute_kbest_cluster_labels(
    df: pd.DataFrame,
    numeric_cols: Sequence[str],
    seed: int,
) -> Tuple[np.ndarray, Dict[str, object]]:
    X = df[list(numeric_cols)].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.values)
    n_comp = int(min(20, Xs.shape[1], Xs.shape[0]))
    pca = PCA(n_components=n_comp, random_state=seed)
    scores = pca.fit_transform(Xs)
    cum = np.cumsum(pca.explained_variance_ratio_)
    n_80 = int(np.searchsorted(cum, 0.80, side="left") + 1) if len(cum) else 2
    n_80 = max(2, min(20, n_80))
    d_fixed = max(2, min(6, scores.shape[1]))

    candidates = [("Dfixed6", d_fixed), ("D80", n_80)]
    best = {"silhouette": -np.inf, "dtag": None, "k": None, "labels": None}

    for dtag, d in candidates:
        Xd = scores[:, :d]
        for k in range(2, 9):
            if k >= Xd.shape[0]:
                continue
            km = KMeans(n_clusters=k, n_init=50, random_state=seed)
            labels = km.fit_predict(Xd)
            sil = float(silhouette_score(Xd, labels))
            if sil > best["silhouette"]:
                best = {"silhouette": sil, "dtag": dtag, "k": k, "labels": labels}

    if best["labels"] is None:
        labels = np.zeros(len(df), dtype=int)
        return labels, {"dtag": "fallback", "k": 1, "silhouette": np.nan}
    return np.asarray(best["labels"], dtype=int), {"dtag": best["dtag"], "k": int(best["k"]), "silhouette": float(best["silhouette"])}


def build_cluster_consequents(df: pd.DataFrame, feature_blocks: Dict[str, Dict[str, List[str]]], seed: int) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, float], Dict[str, object]]:
    clinical_labels, clinical_meta = compute_kbest_cluster_labels(
        df=df,
        numeric_cols=feature_blocks["clinical_only"]["numeric"],
        seed=seed,
    )
    combined_labels, combined_meta = compute_kbest_cluster_labels(
        df=df,
        numeric_cols=feature_blocks["combined"]["numeric"],
        seed=seed,
    )

    clin_items: Dict[str, pd.Series] = {}
    comb_items: Dict[str, pd.Series] = {}
    baseline: Dict[str, float] = {}

    for cl in sorted(np.unique(clinical_labels).tolist()):
        name = f"cluster_clin_kbest_{int(cl)}"
        flag = pd.Series(clinical_labels == cl, index=df.index).astype(bool)
        clin_items[name] = flag
        baseline[name] = float(flag.mean())

    for cl in sorted(np.unique(combined_labels).tolist()):
        name = f"cluster_comb_kbest_{int(cl)}"
        flag = pd.Series(combined_labels == cl, index=df.index).astype(bool)
        comb_items[name] = flag
        baseline[name] = float(flag.mean())

    meta = {
        "clinical_only": clinical_meta,
        "combined": combined_meta,
    }
    return pd.DataFrame(clin_items, index=df.index), pd.DataFrame(comb_items, index=df.index), baseline, meta


def find_numeric_col(df: pd.DataFrame, patterns: Sequence[str]) -> str | None:
    for c in df.columns:
        n = normalize_name(c)
        if "__missing" in c.lower():
            continue
        if any(p in n for p in patterns) and pd.api.types.is_numeric_dtype(df[c]):
            return c
    return None


def build_severity_consequents(
    df: pd.DataFrame,
    ref_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict[str, float], Dict[str, str], Dict[str, float]]:
    spec = {
        "BUN_high": {
            "data_patterns": ["bunconcentration", "bun"],
            "ref_patterns": ["bun"],
            "tail": "high",
        },
        "CREA_high": {
            "data_patterns": ["creaconcentration", "crea"],
            "ref_patterns": ["crea"],
            "tail": "high",
        },
        "leukocyte_high": {
            "data_patterns": ["leukocytecount"],
            "ref_patterns": ["leukocytecount"],
            "tail": "high",
        },
        "neutrophil_high": {
            "data_patterns": ["neutrophilcount"],
            "ref_patterns": ["neutrophilcount"],
            "tail": "high",
        },
        "albumin_low": {
            "data_patterns": ["albuminconcentration", "albumin"],
            "ref_patterns": ["albumin"],
            "tail": "low",
        },
    }

    out: Dict[str, pd.Series] = {}
    baseline: Dict[str, float] = {}
    source_cols: Dict[str, str] = {}
    thresholds_used: Dict[str, float] = {}

    for item, cfg in spec.items():
        col = find_numeric_col(df, cfg["data_patterns"])
        if col is None:
            continue
        ref_param = find_ref_parameter(ref_df, cfg["ref_patterns"])
        if ref_param is None:
            continue
        threshold = extract_threshold_from_ref(ref_df, ref_param, direction=cfg["tail"])
        if threshold is None or not np.isfinite(threshold):
            continue

        s = pd.to_numeric(df[col], errors="coerce")
        if s.notna().sum() == 0:
            continue
        if cfg["tail"] == "high":
            flag = (s >= float(threshold)).fillna(False).astype(bool)
        else:
            flag = (s <= float(threshold)).fillna(False).astype(bool)

        out[item] = flag
        baseline[item] = float(flag.mean())
        source_cols[item] = col
        thresholds_used[item] = float(threshold)

    return pd.DataFrame(out, index=df.index), baseline, source_cols, thresholds_used


def format_antecedents(fs: frozenset) -> str:
    return ", ".join(sorted([str(x) for x in fs]))


def mine_rules_for_consequents(
    tx_env: pd.DataFrame,
    consequent_df: pd.DataFrame,
    baseline_map: Dict[str, float],
    track: str,
    min_support: float,
    min_lift: float,
    singleton_min_lift: float,
    top_n: int = 10,
) -> pd.DataFrame:
    n_rows = len(tx_env)
    antecedent_cols = tx_env.columns.tolist()
    out_rows: List[pd.DataFrame] = []

    for consequent in consequent_df.columns.tolist():
        baseline = float(baseline_map.get(consequent, np.nan))
        if not np.isfinite(baseline) or baseline <= 0:
            continue
        min_conf = baseline

        data = pd.concat([tx_env, consequent_df[[consequent]]], axis=1).fillna(False).astype(bool)
        itemsets = apriori(
            data,
            min_support=min_support,
            use_colnames=True,
            max_len=4,
        )
        if itemsets.empty:
            continue

        rules = association_rules(itemsets, metric="confidence", min_threshold=min_conf)
        if rules.empty:
            continue

        rules = rules[
            rules["antecedents"].apply(lambda s: 1 <= len(s) <= 3 and set(s).issubset(set(antecedent_cols)))
            & rules["consequents"].apply(lambda s: len(s) == 1 and consequent in s)
            & (rules["support"] >= min_support)
            & (rules["confidence"] >= min_conf)
            & (rules["lift"] >= min_lift)
        ].copy()
        if rules.empty:
            continue

        rules["antecedent_len"] = rules["antecedents"].apply(len)
        rules = rules[~((rules["antecedent_len"] == 1) & (rules["lift"] < singleton_min_lift))].copy()
        if rules.empty:
            continue

        rules["track"] = track
        rules["consequent_name"] = consequent
        rules["antecedents"] = rules["antecedents"].apply(format_antecedents)
        rules["support_count"] = (rules["support"] * n_rows).round().astype(int)
        rules["baseline_prevalence"] = baseline
        rules = rules.sort_values(["lift", "confidence"], ascending=[False, False]).head(top_n)

        keep = [
            "track",
            "consequent_name",
            "antecedents",
            "support",
            "support_count",
            "confidence",
            "lift",
            "baseline_prevalence",
        ]
        for optional in ["leverage", "conviction"]:
            if optional in rules.columns:
                keep.append(optional)
        out_rows.append(rules[keep].reset_index(drop=True))

    if not out_rows:
        cols = [
            "track",
            "consequent_name",
            "antecedents",
            "support",
            "support_count",
            "confidence",
            "lift",
            "baseline_prevalence",
            "leverage",
            "conviction",
        ]
        return pd.DataFrame(columns=cols)
    return pd.concat(out_rows, axis=0, ignore_index=True)


def rules_count_by_consequent(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["consequent_name", "n_rules"])
    return (
        df.groupby("consequent_name", dropna=False)
        .size()
        .rename("n_rules")
        .reset_index()
        .sort_values(["n_rules", "consequent_name"], ascending=[False, True])
        .reset_index(drop=True)
    )


def to_md_table(df: pd.DataFrame, cols: Sequence[str]) -> str:
    if df.empty:
        return "_No rows._"
    use_cols = [c for c in cols if c in df.columns]
    lines = []
    lines.append("| " + " | ".join(use_cols) + " |")
    lines.append("| " + " | ".join(["---"] * len(use_cols)) + " |")
    for _, row in df[use_cols].iterrows():
        vals = []
        for c in use_cols:
            v = row[c]
            if isinstance(v, float):
                vals.append(f"{v:.4f}")
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def write_summary(
    out_path: Path,
    env_numeric: Sequence[str],
    env_categorical: Sequence[str],
    baseline_df: pd.DataFrame,
    disease_rules: pd.DataFrame,
    cluster_clin_rules: pd.DataFrame,
    cluster_comb_rules: pd.DataFrame,
    severity_rules: pd.DataFrame,
    cluster_meta: Dict[str, object],
    severity_source_cols: Dict[str, str],
    severity_thresholds: Dict[str, float],
) -> None:
    track_to_df = {
        "disease": disease_rules,
        "cluster_clin": cluster_clin_rules,
        "cluster_comb": cluster_comb_rules,
        "severity": severity_rules,
    }
    all_rules = pd.concat([disease_rules, cluster_clin_rules, cluster_comb_rules, severity_rules], ignore_index=True)
    count_df = rules_count_by_consequent(all_rules)

    lines: List[str] = []
    lines.append("# Apriori Refined Final Summary")
    lines.append("")
    lines.append("## Setup")
    lines.append(f"- Environmental numeric features (kept): {len(env_numeric)}")
    lines.append(f"- Environmental categorical features: {len(env_categorical)}")
    lines.append(f"- Clinical cluster k_best source: clinical_only={cluster_meta['clinical_only']}, combined={cluster_meta['combined']}")
    if severity_thresholds:
        lines.append("- Severity cutoffs were taken from `Heamtology parameter interpreta` only:")
        for k in sorted(severity_thresholds.keys()):
            src = severity_source_cols.get(k, "(column not found)")
            lines.append(f"  - {k}: threshold={severity_thresholds[k]:.4f}, data_column={src}")
    lines.append("")
    lines.append("## Baseline Prevalence By Consequent")
    lines.append(to_md_table(baseline_df, ["track", "consequent_name", "baseline_prevalence"]))
    lines.append("")
    lines.append("## Qualifying Rule Counts By Consequent")
    lines.append(to_md_table(count_df, ["consequent_name", "n_rules"]))
    lines.append("")
    lines.append("## Top 3 Strongest Rules Per Track")
    for track, df in track_to_df.items():
        lines.append(f"### {track}")
        if df.empty:
            lines.append("- No rules met thresholds.")
            continue
        top3 = df.sort_values(["lift", "confidence"], ascending=[False, False]).head(3).copy()
        lines.append(
            to_md_table(
                top3,
                ["consequent_name", "antecedents", "support", "confidence", "lift", "baseline_prevalence"],
            )
        )
    lines.append("")
    lines.append("## Note")
    lines.append("- Exploratory association mining only. These rules are non-causal and hypothesis-generating.")

    out_path.write_text("\n".join(lines), encoding="utf-8")


def main() -> None:
    args = parse_args()
    np.random.seed(args.seed)
    args.outdir.mkdir(parents=True, exist_ok=True)

    df = load_dataset(args.input)
    drop_weather_reason = [c for c in df.columns if "weather_missing_reason" in c.lower()]
    if drop_weather_reason:
        df = df.drop(columns=drop_weather_reason)

    df, outcome_col, feature_blocks, kept_weather_numeric = build_model_aligned_blocks(
        df=df,
        corr_threshold=args.corr_threshold,
    )
    ref_df = load_hematology_interpretation(args.interpretation_workbook, args.interpretation_sheet)
    df = add_lunar_window_flags(df, args.window_days)

    env_numeric = list(feature_blocks["environmental_only"]["numeric"])
    env_numeric_for_items = [
        c for c in env_numeric if normalize_name(c) not in {"moonphasesin", "moonphasecos"}
    ]
    env_categorical = list(feature_blocks["environmental_only"]["categorical"])
    tx_env = build_environmental_items(
        df=df,
        env_numeric_cols=env_numeric_for_items,
        env_categorical_cols=env_categorical,
    )

    disease_df, disease_baseline = build_disease_consequents(df, outcome_col)
    cluster_clin_df, cluster_comb_df, cluster_baseline, cluster_meta = build_cluster_consequents(
        df=df,
        feature_blocks=feature_blocks,
        seed=args.seed,
    )
    severity_df, severity_baseline, severity_source_cols, severity_thresholds = build_severity_consequents(
        df=df,
        ref_df=ref_df,
    )

    disease_rules = mine_rules_for_consequents(
        tx_env=tx_env,
        consequent_df=disease_df,
        baseline_map=disease_baseline,
        track="disease",
        min_support=0.05,
        min_lift=1.25,
        singleton_min_lift=1.5,
        top_n=10,
    )
    cluster_clin_rules = mine_rules_for_consequents(
        tx_env=tx_env,
        consequent_df=cluster_clin_df,
        baseline_map=cluster_baseline,
        track="cluster_clin",
        min_support=0.05,
        min_lift=1.20,
        singleton_min_lift=1.20,
        top_n=10,
    )
    cluster_comb_rules = mine_rules_for_consequents(
        tx_env=tx_env,
        consequent_df=cluster_comb_df,
        baseline_map=cluster_baseline,
        track="cluster_comb",
        min_support=0.05,
        min_lift=1.20,
        singleton_min_lift=1.20,
        top_n=10,
    )
    severity_rules = mine_rules_for_consequents(
        tx_env=tx_env,
        consequent_df=severity_df,
        baseline_map=severity_baseline,
        track="severity",
        min_support=0.05,
        min_lift=1.20,
        singleton_min_lift=1.20,
        top_n=10,
    )

    master = pd.concat(
        [disease_rules, cluster_clin_rules, cluster_comb_rules, severity_rules],
        axis=0,
        ignore_index=True,
    )

    disease_rules.to_csv(args.outdir / "disease_rules.csv", index=False)
    cluster_clin_rules.to_csv(args.outdir / "cluster_rules_clinical_kbest.csv", index=False)
    cluster_comb_rules.to_csv(args.outdir / "cluster_rules_combined_kbest.csv", index=False)
    severity_rules.to_csv(args.outdir / "severity_rules.csv", index=False)
    master.to_csv(args.outdir / "apriori_master_rules.csv", index=False)

    baseline_rows = []
    for k, v in disease_baseline.items():
        baseline_rows.append({"track": "disease", "consequent_name": k, "baseline_prevalence": float(v)})
    for k, v in cluster_baseline.items():
        track = "cluster_clin" if k.startswith("cluster_clin_") else "cluster_comb"
        baseline_rows.append({"track": track, "consequent_name": k, "baseline_prevalence": float(v)})
    for k, v in severity_baseline.items():
        baseline_rows.append({"track": "severity", "consequent_name": k, "baseline_prevalence": float(v)})
    baseline_df = pd.DataFrame(baseline_rows).sort_values(["track", "consequent_name"]).reset_index(drop=True)

    write_summary(
        out_path=args.outdir / "apriori_summary.md",
        env_numeric=env_numeric,
        env_categorical=env_categorical,
        baseline_df=baseline_df,
        disease_rules=disease_rules,
        cluster_clin_rules=cluster_clin_rules,
        cluster_comb_rules=cluster_comb_rules,
        severity_rules=severity_rules,
        cluster_meta=cluster_meta,
        severity_source_cols=severity_source_cols,
        severity_thresholds=severity_thresholds,
    )

    track_map = {
        "disease": disease_rules,
        "cluster_clin": cluster_clin_rules,
        "cluster_comb": cluster_comb_rules,
        "severity": severity_rules,
    }
    print("=== Apriori Refined Final ===")
    print(f"Output directory: {args.outdir}")
    print(f"Environmental numeric kept: {len(kept_weather_numeric)}")
    print(f"Environmental item columns: {tx_env.shape[1]}")
    print(f"Interpretation workbook/sheet: {args.interpretation_workbook} :: {args.interpretation_sheet}")
    for track, rdf in track_map.items():
        print(f"{track}: {len(rdf)} rules")
        if not rdf.empty:
            top = rdf.sort_values(["lift", "confidence"], ascending=[False, False]).iloc[0]
            print(
                f"  strongest => {top['consequent_name']} <= {top['antecedents']} "
                f"(lift={top['lift']:.3f}, conf={top['confidence']:.3f}, support={top['support']:.3f})"
            )
        else:
            print("  strongest => none (no rule met threshold)")


if __name__ == "__main__":
    main()
